[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/14.%20The%20Vessel%20Stowage%20Planning%20Problem%20%28Export%29/P14-Tier-11_executed.ipynb)

---


# The Vessel Stowage Planning Problem (Export) — Tier 11
## Physical-Cyber Synthesis: Programmable and Digital Matter

### Purpose and scope

This notebook presents an illustrative, rule-based simulation of a vessel and containers that can change their modeled properties. It follows a procedural sequence: initialize a vessel and container ecosystem, apply container configuration rules, adapt the vessel to environmental conditions, simulate cargo-specific reconfiguration, and summarize the resulting metrics.

The code demonstrates how state, rules, and metric calculations can be organized in a simulation. It does **not** solve a conventional vessel stowage optimization problem: there are no container-to-slot assignment decisions, no mathematical optimizer, and no proof of optimality. The procedures are illustrative heuristics rather than a validated engineering or operational model.

### Model assumptions

- Container and bay dimensions are abstract model length units. Their ratios are used for the space-utilization calculation; the values should not be interpreted as verified vessel or container specifications.
- Energy costs, energy recovery, waste, and recycling values are illustrative model quantities. They are not measured physical quantities or independently validated forecasts.
- The vessel profile is an example inspired by large container vessels. The capacity parameter is a scenario input, not a verified specification for a named vessel.
- Container weights and environmental conditions are sampled randomly. No random seed is set, so exact values can vary between runs.
- The rules do not represent naval architecture, material science, safety certification, cargo compatibility, or real-time control. They are included to make a cyber-physical simulation structure concrete.

### How to read the model

Each object stores a **state**: the values describing it at a particular point in the simulation. Methods such as `reshape` and `subdivide` change that state and return a modeled cost. The synthesis system applies those operations in a fixed sequence and computes summary metrics from the resulting states. The score is a weighted indicator defined by this notebook, not an objective value from a stowage optimization model.

## Model and procedural rules

### State changes and modeled energy

A programmable material has a current state and a set of numerical properties. A transformation changes the state and any requested properties. In the code, `ProgrammableMatter.transform` calculates a simplified transformation cost and then updates the material object.

For a requested target state \(s'\) and target properties \(P'\), the modeled cost is

$$
E = \frac{100\,|\operatorname{ord}(s'_0)-\operatorname{ord}(s_0)| + \sum_{p\in P'} |p'-p|}{\eta},
$$

where \(s_0\) is the current state, \(s'_0\) is the first character of the target state, \(p\) and \(p'\) are the current and target values of a requested property, and \(\eta\) is the material's energy-efficiency parameter. This expression is an illustrative rule; its result is reported in **model energy units**, not megajoules.

### Container configuration rule

For each container, the model computes the weight-to-volume ratio

$$
r = \frac{w}{1000V},
$$

where \(w\) is weight in the model's stated weight units and \(V\) is modeled volume. If \(r < 0.3\), the container is reshaped using the dimension multipliers \(0.8\), \(0.8\), and \(1.5625\). Their product is \(1\), so the modeled volume is preserved. If the container's weight exceeds 20 and it has no compartments, it is subdivided into four compartments. These are sequential rules, not a search over alternative configurations.

The aggregate space-utilization metric is

$$
U = \frac{\sum_{c\in C} V_c}{\sum_{b\in B} L_bW_bH_b},
$$

where \(C\) is the set of modeled containers, \(B\) is the set of vessel bays, \(V_c\) is container volume, and \(L_b,W_b,H_b\) are the dimensions of bay \(b\). Because reshaping preserves each container's volume and subdivision does not change it, these operations alone do not increase \(U\). This metric is a volume ratio, not a measure of feasible packing or slot assignment.

### Vessel adaptation rule

The environmental conditions are checked in a fixed order. If wave height exceeds 3.0, the two end bays are adjusted. Otherwise, if wind speed exceeds 20.0, the middle bays are adjusted. The stability metric compares the average bay dimensions with reference dimensions and returns a bounded score in \([0,1]\). It is a simplified indicator, not a naval-architecture stability calculation.

### Summary score

The score combines normalized space, stability, energy, waste, and efficiency indicators:

$$
S = 0.3S_U + 0.2S_B + 0.2S_E + 0.2S_W + 0.1S_G.
$$

The terms are computed in `_calculate_synthesis_score`. Positive changes contribute to the space and stability terms; energy cost reduces the energy term. This weighted score is only a descriptive index for this scenario. It is not a validated performance measure and does not establish that a plan is optimal.

In [ ]:
from dataclasses import dataclass, field, replace
from datetime import datetime
from typing import Any, Dict, List, Tuple
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8")


In [ ]:
@dataclass
class ProgrammableMatter:
    """Store illustrative material properties and a mutable material state."""

    id: str
    base_material: str
    current_state: str
    density: float
    stiffness: float
    conductivity: float
    programmability: float
    energy_efficiency: float

    def transform(
        self,
        target_state: str,
        target_properties: Dict[str, float],
    ) -> float:
        """Update the material state and return its modeled transformation cost."""
        if not target_state:
            raise ValueError("target_state must not be empty")
        if self.energy_efficiency <= 0:
            raise ValueError("energy_efficiency must be positive")

        state_change_energy = abs(
            ord(target_state[0]) - ord(self.current_state[0])
        ) * 100
        property_changes = 0.0
        for prop, target_value in target_properties.items():
            if not hasattr(self, prop):
                raise AttributeError(f"Unknown material property: {prop}")
            current_value = getattr(self, prop)
            property_changes += abs(target_value - current_value)

        total_energy = (
            state_change_energy + property_changes
        ) / self.energy_efficiency

        self.current_state = target_state
        for prop, target_value in target_properties.items():
            setattr(self, prop, target_value)

        return total_energy


@dataclass
class ProgrammableContainer:
    """Store container geometry, cargo state, and reconfiguration history."""

    id: str
    base_matter: ProgrammableMatter
    current_length: float
    current_width: float
    current_height: float
    current_volume: float
    current_weight: float
    internal_compartments: List[Dict[str, Any]] = field(default_factory=list)
    cargo_adaptation: Dict[str, Any] = field(default_factory=dict)
    reconfiguration_history: List[Dict[str, Any]] = field(default_factory=list)

    def reshape(
        self,
        target_length: float,
        target_width: float,
        target_height: float,
    ) -> float:
        """Reshape the container and record the modeled cost and dimensions."""
        if min(target_length, target_width, target_height) <= 0:
            raise ValueError("Target dimensions must be positive")

        old_dimensions = (
            self.current_length,
            self.current_width,
            self.current_height,
        )
        old_volume = self.current_volume
        new_volume = target_length * target_width * target_height
        target_properties = {
            "density": self.base_matter.density * (old_volume / new_volume),
            "stiffness": self.base_matter.stiffness * 0.9,
        }
        energy_cost = self.base_matter.transform("solid", target_properties)

        self.current_length = target_length
        self.current_width = target_width
        self.current_height = target_height
        self.current_volume = new_volume
        self.reconfiguration_history.append(
            {
                "timestamp": datetime.now(),
                "type": "reshape",
                "old_dimensions": old_dimensions,
                "new_dimensions": (
                    target_length,
                    target_width,
                    target_height,
                ),
                "old_volume": old_volume,
                "new_volume": new_volume,
                "energy_cost": energy_cost,
            }
        )
        return energy_cost

    def subdivide(
        self,
        n_compartments: int,
        compartment_types: List[str],
    ) -> float:
        """Create modeled compartments and return their modeled energy cost."""
        if n_compartments <= 1:
            return 0.0
        if not compartment_types:
            raise ValueError("At least one compartment type is required")
        if self.base_matter.programmability <= 0:
            raise ValueError("programmability must be positive")

        subdivision_energy = (
            n_compartments * 50 / self.base_matter.programmability
        )
        compartment_volume = self.current_volume / n_compartments
        self.internal_compartments = []

        for index in range(n_compartments):
            self.internal_compartments.append(
                {
                    "id": f"comp_{index + 1}",
                    "type": compartment_types[index % len(compartment_types)],
                    "volume": compartment_volume,
                    "occupancy": 0.0,
                    "environment": {
                        "temperature": 20.0,
                        "humidity": 50.0,
                        "pressure": 101.325,
                    },
                }
            )

        self.reconfiguration_history.append(
            {
                "timestamp": datetime.now(),
                "type": "subdivide",
                "n_compartments": n_compartments,
                "energy_cost": subdivision_energy,
            }
        )
        return subdivision_energy

    def redistribute_weight(
        self,
        target_cog: Tuple[float, float, float],
    ) -> float:
        """Estimate a modeled cost for moving cargo toward a target center of gravity."""
        if self.base_matter.programmability <= 0:
            raise ValueError("programmability must be positive")

        current_cog = (
            self.current_length / 2,
            self.current_width / 2,
            self.current_height / 2,
        )
        cog_distance = math.sqrt(
            sum((target - current) ** 2
                for target, current in zip(target_cog, current_cog))
        )
        redistribution_energy = (
            cog_distance * self.current_weight * 10
            / self.base_matter.programmability
        )
        self.cargo_adaptation["target_cog"] = target_cog
        self.cargo_adaptation["current_cog"] = target_cog
        return redistribution_energy


@dataclass
class ProgrammableVessel:
    """Store the illustrative vessel configuration and adaptive-system state."""

    id: str
    base_capacity: int
    programmable_bays: List[Dict[str, Any]] = field(default_factory=list)
    morphing_capabilities: Dict[str, Any] = field(default_factory=dict)
    adaptive_systems: Dict[str, Any] = field(default_factory=dict)
    real_time_metrics: Dict[str, float] = field(default_factory=dict)

    def __post_init__(self) -> None:
        if not self.programmable_bays:
            for bay_number in range(1, 23):
                self.programmable_bays.append(
                    {
                        "id": f"bay_{bay_number}",
                        "current_length": 12.0,
                        "current_width": 32.0,
                        "current_height": 28.0,
                        "current_capacity": 100,
                        "morphing_state": "standard",
                        "programmability": 0.8,
                        "energy_usage": 0.0,
                    }
                )

        if not self.morphing_capabilities:
            self.morphing_capabilities = {
                "length_range": (8.0, 20.0),
                "width_range": (24.0, 40.0),
                "height_range": (20.0, 35.0),
                "morphing_speed": 0.1,
                "energy_efficiency": 0.7,
            }

        if not self.adaptive_systems:
            self.adaptive_systems = {
                "hull_integrity": {
                    "current": 0.95,
                    "target": 0.98,
                    "adaptive": True,
                },
                "stability_control": {
                    "current": 0.92,
                    "target": 0.96,
                    "adaptive": True,
                },
                "ballast_system": {
                    "current": 0.88,
                    "target": 0.94,
                    "adaptive": True,
                },
                "power_system": {
                    "current": 0.91,
                    "target": 0.97,
                    "adaptive": True,
                },
            }


@dataclass
class PhysicalCyberSynthesis:
    """Coordinate the rule-based physical-cyber simulation."""

    vessel: ProgrammableVessel
    programmable_containers: List[ProgrammableContainer] = field(default_factory=list)
    digital_twin_state: Dict[str, Any] = field(default_factory=dict)
    synthesis_history: List[Dict[str, Any]] = field(default_factory=list)
    performance_metrics: Dict[str, Any] = field(default_factory=dict)

    def initialize_programmable_ecosystem(
        self,
        n_containers: int,
    ) -> Dict[str, Any]:
        """Create modeled containers and capture their initial digital-twin state."""
        if n_containers < 0:
            raise ValueError("n_containers must not be negative")

        print(f"Initializing programmable ecosystem with {n_containers} containers...")

        matter_templates = [
            ProgrammableMatter(
                "PM_001", "graphene", "solid", 2267, 1000, 5000, 0.9, 0.8
            ),
            ProgrammableMatter(
                "PM_002", "carbon_nanotube", "solid", 1600, 1200, 3500, 0.85, 0.75
            ),
            ProgrammableMatter(
                "PM_003", "shape_memory_alloy", "solid", 6450, 70, 20, 0.7, 0.6
            ),
            ProgrammableMatter(
                "PM_004", "electroactive_polymer", "gel", 1200, 2, 0.2, 0.95, 0.9
            ),
        ]
        container_specs = [
            (40, 2.4, 2.4, 2.6),
            (45, 2.4, 2.4, 2.9),
            (20, 2.4, 2.4, 1.3),
            (48, 2.6, 2.6, 2.8),
        ]

        self.programmable_containers = []
        for index in range(n_containers):
            spec_index = index % len(container_specs)
            _nominal_type, length, width, height = container_specs[spec_index]
            template = matter_templates[index % len(matter_templates)]
            matter = replace(
                template,
                id=f"{template.id}_{index + 1:04d}",
            )
            container = ProgrammableContainer(
                id=f"PC_{index + 1:04d}",
                base_matter=matter,
                current_length=length,
                current_width=width,
                current_height=height,
                current_volume=length * width * height,
                current_weight=15 + random.uniform(-5, 25),
            )
            self.programmable_containers.append(container)

        self.digital_twin_state = {
            "timestamp": datetime.now(),
            "vessel_state": self._capture_vessel_state(),
            "container_states": [
                self._capture_container_state(container)
                for container in self.programmable_containers
            ],
            "environmental_conditions": self._get_environmental_conditions(),
            "synthesis_mode": "adaptive",
            "energy_usage": 0.0,
        }

        material_types = sorted(
            {
                container.base_matter.base_material
                for container in self.programmable_containers
            }
        )
        mean_programmability = (
            float(
                np.mean(
                    [
                        container.base_matter.programmability
                        for container in self.programmable_containers
                    ]
                )
            )
            if self.programmable_containers
            else 0.0
        )
        return {
            "n_containers": len(self.programmable_containers),
            "matter_types": material_types,
            "total_programmability": mean_programmability,
            "ecosystem_initialized": True,
        }

    def _capture_vessel_state(self) -> Dict[str, Any]:
        """Return a snapshot of vessel bays and adaptive systems."""
        return {
            "bay_configurations": [
                bay.copy() for bay in self.vessel.programmable_bays
            ],
            "adaptive_systems": {
                name: system.copy()
                for name, system in self.vessel.adaptive_systems.items()
            },
            "total_capacity": sum(
                bay["current_capacity"] for bay in self.vessel.programmable_bays
            ),
            "morphing_energy": sum(
                bay["energy_usage"] for bay in self.vessel.programmable_bays
            ),
        }

    def _capture_container_state(
        self,
        container: ProgrammableContainer,
    ) -> Dict[str, Any]:
        """Return a snapshot of one container's modeled state."""
        return {
            "id": container.id,
            "dimensions": (
                container.current_length,
                container.current_width,
                container.current_height,
            ),
            "volume": container.current_volume,
            "weight": container.current_weight,
            "compartments": len(container.internal_compartments),
            "matter_state": container.base_matter.current_state,
            "programmability": container.base_matter.programmability,
        }

    def _get_environmental_conditions(self) -> Dict[str, float]:
        """Sample illustrative environmental conditions for the scenario."""
        return {
            "temperature": random.uniform(15, 35),
            "humidity": random.uniform(40, 80),
            "wind_speed": random.uniform(0, 25),
            "wave_height": random.uniform(0, 5),
            "sea_state": random.uniform(0, 6),
        }

    def optimize_space_utilization(self) -> Dict[str, Any]:
        """Apply the stated per-container rules and report volume utilization."""
        if not self.vessel.programmable_bays:
            raise ValueError("The vessel must contain at least one bay")

        print("\n=== Space Utilization Configuration ===")
        results = {
            "initial_utilization": 0.0,
            "final_utilization": 0.0,
            "container_transformations": [],
            "energy_cost": 0.0,
            "time_cost": 0.0,
        }
        start_time = time.time()
        vessel_volume = sum(
            bay["current_length"]
            * bay["current_width"]
            * bay["current_height"]
            for bay in self.vessel.programmable_bays
        )
        if vessel_volume <= 0:
            raise ValueError("Total vessel volume must be positive")

        total_volume = sum(
            container.current_volume
            for container in self.programmable_containers
        )
        initial_utilization = total_volume / vessel_volume
        results["initial_utilization"] = initial_utilization
        print(f"Initial modeled space utilization: {initial_utilization:.2%}")

        for container in self.programmable_containers:
            optimal_config = self._determine_optimal_container_config(container)

            if optimal_config["reshape"]:
                old_dimensions = (
                    container.current_length,
                    container.current_width,
                    container.current_height,
                )
                energy_cost = container.reshape(
                    optimal_config["length"],
                    optimal_config["width"],
                    optimal_config["height"],
                )
                results["energy_cost"] += energy_cost
                results["container_transformations"].append(
                    {
                        "container_id": container.id,
                        "type": "reshape",
                        "old_dimensions": old_dimensions,
                        "new_dimensions": (
                            container.current_length,
                            container.current_width,
                            container.current_height,
                        ),
                        "energy_cost": energy_cost,
                    }
                )

            if optimal_config["subdivide"]:
                energy_cost = container.subdivide(
                    optimal_config["n_compartments"],
                    optimal_config["compartment_types"],
                )
                results["energy_cost"] += energy_cost
                results["container_transformations"].append(
                    {
                        "container_id": container.id,
                        "type": "subdivide",
                        "n_compartments": optimal_config["n_compartments"],
                        "energy_cost": energy_cost,
                    }
                )

        total_volume = sum(
            container.current_volume
            for container in self.programmable_containers
        )
        final_utilization = total_volume / vessel_volume
        results["final_utilization"] = final_utilization
        results["time_cost"] = time.time() - start_time
        print(f"Final modeled space utilization: {final_utilization:.2%}")
        if not math.isclose(initial_utilization, 0.0):
            relative_change = (
                (final_utilization - initial_utilization)
                / initial_utilization
                * 100
            )
            print(f"Relative change: {relative_change:+.1f}%")
        print(f"Total modeled energy cost: {results['energy_cost']:.2f} units")
        return results

    def _determine_optimal_container_config(
        self,
        container: ProgrammableContainer,
    ) -> Dict[str, Any]:
        """Apply the notebook's fixed threshold rules to one container."""
        if container.current_volume <= 0:
            raise ValueError("Container volume must be positive")

        current_utilization = container.current_weight / (
            container.current_volume * 1000
        )
        config = {
            "reshape": False,
            "subdivide": False,
            "length": container.current_length,
            "width": container.current_width,
            "height": container.current_height,
            "n_compartments": 0,
            "compartment_types": [],
        }

        if current_utilization < 0.3:
            config["reshape"] = True
            config["length"] = container.current_length * 0.8
            config["width"] = container.current_width * 0.8
            config["height"] = container.current_height * 1.5625

        if (
            container.current_weight > 20
            and len(container.internal_compartments) == 0
        ):
            config["subdivide"] = True
            config["n_compartments"] = 4
            config["compartment_types"] = [
                "dry",
                "refrigerated",
                "hazardous",
                "valuable",
            ]
        return config

    def optimize_vessel_structure(
        self,
        environmental_conditions: Dict[str, float],
    ) -> Dict[str, Any]:
        """Apply the vessel's fixed environmental adaptation rules."""
        required_conditions = {"wave_height", "wind_speed"}
        missing_conditions = required_conditions - environmental_conditions.keys()
        if missing_conditions:
            raise ValueError(
                f"Missing environmental conditions: {sorted(missing_conditions)}"
            )

        print("\n=== Vessel Structure Adaptation ===")
        results = {
            "initial_stability": 0.0,
            "final_stability": 0.0,
            "bay_transformations": [],
            "energy_cost": 0.0,
            "adaptations": [],
        }
        initial_stability = self._calculate_vessel_stability()
        results["initial_stability"] = initial_stability
        print(f"Initial modeled stability: {initial_stability:.3f}")

        if environmental_conditions["wave_height"] > 3.0:
            print("Applying the rough-sea adaptation rule...")
            for bay in self.vessel.programmable_bays:
                if bay["id"] in {"bay_1", "bay_22"}:
                    bay["current_length"] *= 0.95
                    bay["current_width"] *= 1.05
                    bay["energy_usage"] += 50
                    results["bay_transformations"].append(
                        {
                            "bay_id": bay["id"],
                            "type": "hull_reinforcement",
                            "energy_cost": 50,
                        }
                    )
            results["adaptations"].append("rough_seas_adaptation")
        elif environmental_conditions["wind_speed"] > 20.0:
            print("Applying the high-wind adaptation rule...")
            for bay in self.vessel.programmable_bays[5:17]:
                bay["current_height"] *= 0.9
                bay["energy_usage"] += 30
                results["bay_transformations"].append(
                    {
                        "bay_id": bay["id"],
                        "type": "sail_area_reduction",
                        "energy_cost": 30,
                    }
                )
            results["adaptations"].append("high_wind_adaptation")

        final_stability = self._calculate_vessel_stability()
        results["final_stability"] = final_stability
        results["energy_cost"] = sum(
            transformation["energy_cost"]
            for transformation in results["bay_transformations"]
        )
        print(f"Final modeled stability: {final_stability:.3f}")
        if not math.isclose(initial_stability, 0.0):
            relative_change = (
                (final_stability - initial_stability)
                / abs(initial_stability)
                * 100
            )
            print(f"Relative change: {relative_change:+.1f}%")
        print(f"Total modeled energy cost: {results['energy_cost']:.2f} units")
        return results

    def _calculate_vessel_stability(self) -> float:
        """Calculate the notebook's bounded, dimension-ratio stability indicator."""
        bay_count = len(self.vessel.programmable_bays)
        if bay_count == 0:
            raise ValueError("The vessel must contain at least one bay")

        total_length = sum(
            bay["current_length"] for bay in self.vessel.programmable_bays
        )
        total_width = sum(
            bay["current_width"] for bay in self.vessel.programmable_bays
        )
        total_height = sum(
            bay["current_height"] for bay in self.vessel.programmable_bays
        )
        length_ratio = total_length / bay_count / 12.0
        width_ratio = total_width / bay_count / 32.0
        height_ratio = total_height / bay_count / 28.0
        stability = (
            1.0
            - abs(length_ratio - 1.0) * 0.3
            - abs(width_ratio - 1.0) * 0.3
            - abs(height_ratio - 1.0) * 0.4
        )
        return max(0.0, min(1.0, stability))

    def simulate_zero_waste_logistics(self) -> Dict[str, Any]:
        """Apply cargo-specific reconfiguration rules to up to ten containers."""
        print("\n=== Illustrative Waste and Reconfiguration Simulation ===")
        results = {
            "containers_reconfigured": 0,
            "waste_eliminated": 0.0,
            "energy_recovered": 0.0,
            "material_recycled": 0.0,
            "efficiency_gains": [],
        }
        destinations = [
            "electronics",
            "textiles",
            "automotive",
            "food",
            "machinery",
        ]

        for index, container in enumerate(self.programmable_containers[:10]):
            destination = destinations[index % len(destinations)]
            print(f"Reconfiguring {container.id} for {destination} cargo...")

            if destination == "electronics":
                container.subdivide(
                    6,
                    [
                        "shock_absorbing",
                        "temperature_controlled",
                        "anti_static",
                        "moisture_barrier",
                        "security",
                        "monitoring",
                    ],
                )
                results["efficiency_gains"].append("electronics_protection")
            elif destination == "automotive":
                container.reshape(12.0, 3.0, 2.8)
                container.subdivide(
                    4,
                    [
                        "parts_racking",
                        "tool_storage",
                        "fluid_containment",
                        "access",
                    ],
                )
                results["efficiency_gains"].append("automotive_optimization")
            elif destination == "food":
                container.subdivide(
                    8,
                    [
                        "refrigerated",
                        "frozen",
                        "dry_storage",
                        "ventilated",
                        "humidity_controlled",
                        "sanitized",
                        "traceability",
                        "quality_monitoring",
                    ],
                )
                results["efficiency_gains"].append("food_safety")
            results["containers_reconfigured"] += 1

        traditional_waste = len(self.programmable_containers) * 500
        modeled_waste = len(self.programmable_containers) * 50
        results["waste_eliminated"] = traditional_waste - modeled_waste
        results["energy_recovered"] = results["containers_reconfigured"] * 100
        results["material_recycled"] = 0.8

        print("\nIllustrative simulation results:")
        print(f"  Containers reconfigured: {results['containers_reconfigured']}")
        print(f"  Modeled waste avoided: {results['waste_eliminated']:.0f} model kg")
        print(f"  Modeled energy recovered: {results['energy_recovered']:.0f} units")
        print(f"  Assumed recycling rate: {results['material_recycled']:.1%}")
        print(f"  Recorded efficiency indicators: {len(results['efficiency_gains'])}")
        return results

    def run_comprehensive_synthesis(self) -> Dict[str, Any]:
        """Run ecosystem initialization, configuration, adaptation, and reporting."""
        print("=== Physical-Cyber Synthesis: Comprehensive Scenario ===")
        results = {
            "space_optimization": {},
            "vessel_optimization": {},
            "zero_waste_simulation": {},
            "overall_metrics": {},
            "synthesis_score": 0.0,
        }

        if not self.programmable_containers:
            self.initialize_programmable_ecosystem(50)

        results["space_optimization"] = self.optimize_space_utilization()
        environmental_conditions = self._get_environmental_conditions()
        results["vessel_optimization"] = self.optimize_vessel_structure(
            environmental_conditions
        )
        results["zero_waste_simulation"] = self.simulate_zero_waste_logistics()
        results["overall_metrics"] = self._calculate_overall_metrics(results)
        results["synthesis_score"] = self._calculate_synthesis_score(results)
        self.synthesis_history.append(results)
        self.performance_metrics = results["overall_metrics"].copy()
        return results

    def _calculate_overall_metrics(
        self,
        results: Dict[str, Any],
    ) -> Dict[str, float]:
        """Combine the component results without changing their interpretation."""
        space_improvement = (
            results["space_optimization"]["final_utilization"]
            - results["space_optimization"]["initial_utilization"]
        )
        stability_improvement = (
            results["vessel_optimization"]["final_stability"]
            - results["vessel_optimization"]["initial_stability"]
        )
        total_energy = (
            results["space_optimization"]["energy_cost"]
            + results["vessel_optimization"]["energy_cost"]
        )
        waste_results = results["zero_waste_simulation"]
        return {
            "space_utilization_improvement": space_improvement,
            "stability_improvement": stability_improvement,
            "total_energy_cost": total_energy,
            "waste_eliminated_kg": waste_results["waste_eliminated"],
            "energy_recovered_mj": waste_results["energy_recovered"],
            "containers_reconfigured": waste_results["containers_reconfigured"],
            "efficiency_improvements": len(waste_results["efficiency_gains"]),
        }

    def _calculate_synthesis_score(self, results: Dict[str, Any]) -> float:
        """Compute the notebook's bounded, illustrative weighted performance index."""
        metrics = results["overall_metrics"]
        space_score = (
            min(1.0, max(0.0, metrics["space_utilization_improvement"] * 10))
            * 0.3
        )
        stability_score = (
            min(1.0, max(0.0, metrics["stability_improvement"] * 10))
            * 0.2
        )
        energy_score = (
            max(0.0, 1.0 - metrics["total_energy_cost"] / 10000) * 0.2
        )
        waste_score = (
            min(1.0, max(0.0, metrics["waste_eliminated_kg"] / 10000))
            * 0.2
        )
        efficiency_score = (
            min(1.0, max(0.0, metrics["efficiency_improvements"] / 20))
            * 0.1
        )
        return space_score + stability_score + energy_score + waste_score + efficiency_score


## Concrete implementation example

The next cell creates an illustrative 24,000-TEU scenario vessel, then initializes 50 modeled containers. The capacity is a scenario parameter; it should not be read as a verified specification for a particular ship. The returned ecosystem summary describes the initialized model state.

In [ ]:
print("=== Physical-Cyber Synthesis System Initialization ===")

programmable_vessel = ProgrammableVessel(
    id="PCS_GULSON_001",
    base_capacity=24000,
)

print("Programmable vessel scenario:")
print(f"  Vessel ID: {programmable_vessel.id}")
print(f"  Scenario capacity: {programmable_vessel.base_capacity:,} TEU")
print(f"  Programmable bays: {len(programmable_vessel.programmable_bays)}")
print(f"  Morphing capabilities: {programmable_vessel.morphing_capabilities}")
print(f"  Adaptive systems: {list(programmable_vessel.adaptive_systems)}")

synthesis_system = PhysicalCyberSynthesis(programmable_vessel)
print("\nSynthesis system initialized.")
print(f"  Vessel: {synthesis_system.vessel.id}")
print(f"  Digital-twin components before ecosystem initialization: {len(synthesis_system.digital_twin_state)}")

ecosystem_results = synthesis_system.initialize_programmable_ecosystem(50)
print("\nProgrammable ecosystem summary:")
for key, value in ecosystem_results.items():
    print(f"  {key.replace('_', ' ').title()}: {value}")

print("\nSample modeled containers:")
for container in synthesis_system.programmable_containers[:3]:
    print(f"  {container.id}:")
    print(f"    Base material: {container.base_matter.base_material}")
    print(
        "    Model dimensions: "
        f"{container.current_length:.1f} × "
        f"{container.current_width:.1f} × "
        f"{container.current_height:.1f} model units"
    )
    print(f"    Modeled volume: {container.current_volume:.1f} cubic model units")
    print(f"    Weight input: {container.current_weight:.1f} model weight units")
    print(f"    Programmability parameter: {container.base_matter.programmability:.1%}")
    print(f"    Compartments: {len(container.internal_compartments)}")


## Run the comprehensive scenario

The synthesis method applies the components in sequence: container configuration, vessel adaptation under newly sampled environmental conditions, and cargo-specific reconfiguration. It then calculates summary metrics and the weighted index. The container ecosystem initialized above is used directly, so this run does not silently add a second set of containers.

In [ ]:
synthesis_results = synthesis_system.run_comprehensive_synthesis()

space_results = synthesis_results["space_optimization"]
vessel_results = synthesis_results["vessel_optimization"]
waste_results = synthesis_results["zero_waste_simulation"]
overall_metrics = synthesis_results["overall_metrics"]
synthesis_score = synthesis_results["synthesis_score"]

print("\n=== Scenario Results ===")
print("\nModeled space utilization:")
print(f"  Initial: {space_results['initial_utilization']:.2%}")
print(f"  Final: {space_results['final_utilization']:.2%}")
print(
    "  Absolute change: "
    f"{space_results['final_utilization'] - space_results['initial_utilization']:+.4%}"
)
print(f"  Container transformation records: {len(space_results['container_transformations'])}")
print(f"  Modeled energy cost: {space_results['energy_cost']:.2f} units")
print(f"  Runtime: {space_results['time_cost']:.4f} seconds")

print("\nModeled vessel stability indicator:")
print(f"  Initial: {vessel_results['initial_stability']:.3f}")
print(f"  Final: {vessel_results['final_stability']:.3f}")
print(
    "  Absolute change: "
    f"{vessel_results['final_stability'] - vessel_results['initial_stability']:+.3f}"
)
print(f"  Bay transformation records: {len(vessel_results['bay_transformations'])}")
print(f"  Modeled energy cost: {vessel_results['energy_cost']:.2f} units")
print(f"  Adaptations: {', '.join(vessel_results['adaptations']) or 'None'}")

print("\nIllustrative waste and reconfiguration metrics:")
print(f"  Containers processed: {waste_results['containers_reconfigured']}")
print(f"  Modeled waste avoided: {waste_results['waste_eliminated']:.0f} model kg")
print(f"  Modeled energy recovered: {waste_results['energy_recovered']:.0f} units")
print(f"  Assumed recycling rate: {waste_results['material_recycled']:.1%}")
print(f"  Recorded efficiency indicators: {len(waste_results['efficiency_gains'])}")

print("\nCombined metrics:")
for metric, value in overall_metrics.items():
    print(f"  {metric.replace('_', ' ').title()}: {value}")

print(f"\nIllustrative synthesis score: {synthesis_score:.3f} (bounded from 0 to 1)")
if synthesis_score > 0.8:
    status = "High under this notebook's illustrative scoring rule"
elif synthesis_score > 0.6:
    status = "Moderate-high under this notebook's illustrative scoring rule"
elif synthesis_score > 0.4:
    status = "Moderate under this notebook's illustrative scoring rule"
else:
    status = "Low under this notebook's illustrative scoring rule"
print(f"  Interpretation: {status}")


## Results dashboard

The charts display the values produced by the current run. The panels use separate units, so bar heights should be compared only within each panel. In particular, the space-utilization panel reports a volume ratio, while the energy and waste panels show illustrative model quantities.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(
    "Physical-Cyber Synthesis: Illustrative Scenario Metrics",
    fontsize=16,
    fontweight="bold",
)

space_values = [
    space_results["initial_utilization"],
    space_results["final_utilization"],
]
axes[0, 0].bar(["Initial", "Final"], space_values, color=["#4C78A8", "#F58518"])
axes[0, 0].set_title("Modeled Space Utilization")
axes[0, 0].set_ylabel("Volume ratio")
axes[0, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.1%}"))
axes[0, 0].grid(axis="y", alpha=0.3)

stability_values = [
    vessel_results["initial_stability"],
    vessel_results["final_stability"],
]
axes[0, 1].bar(["Initial", "Final"], stability_values, color=["#4C78A8", "#F58518"])
axes[0, 1].set_title("Modeled Stability Indicator")
axes[0, 1].set_ylabel("Indicator (0 to 1)")
axes[0, 1].set_ylim(0, 1)
axes[0, 1].grid(axis="y", alpha=0.3)

transformation_types = ["Reshape", "Subdivide"]
transformation_counts = [
    sum(
        record["type"] == transformation_type
        for record in space_results["container_transformations"]
    )
    for transformation_type in transformation_types
]
axes[0, 2].bar(transformation_types, transformation_counts, color=["#72B7B2", "#E45756"])
axes[0, 2].set_title("Space-Stage Transformation Records")
axes[0, 2].set_ylabel("Number of records")
axes[0, 2].grid(axis="y", alpha=0.3)

energy_values = [
    space_results["energy_cost"],
    vessel_results["energy_cost"],
    waste_results["energy_recovered"],
]
energy_labels = ["Container", "Vessel", "Recovered"]
axes[1, 0].bar(energy_labels, energy_values, color=["#E45756", "#F2CF5B", "#54A24B"])
axes[1, 0].set_title("Illustrative Energy Quantities")
axes[1, 0].set_ylabel("Model energy units")
axes[1, 0].grid(axis="y", alpha=0.3)

axes[1, 1].bar(
    ["Waste avoided"],
    [waste_results["waste_eliminated"]],
    color="#54A24B",
)
axes[1, 1].set_title("Illustrative Waste Metric")
axes[1, 1].set_ylabel("Model kg")
axes[1, 1].grid(axis="y", alpha=0.3)

axes[1, 2].bar(
    ["Recycling rate"],
    [waste_results["material_recycled"]],
    color="#4C78A8",
)
axes[1, 2].set_title("Assumed Recycling Rate")
axes[1, 2].set_ylabel("Fraction")
axes[1, 2].set_ylim(0, 1)
axes[1, 2].yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
axes[1, 2].grid(axis="y", alpha=0.3)

for axis in axes.flat:
    axis.tick_params(axis="x", labelrotation=15)

fig.tight_layout()
plt.show()

print("Dashboard created from the current scenario results.")


## Interpreting the results

- A space-utilization change near zero is consistent with the modeled rules: the space-stage reshape preserves container volume, and subdivision does not change it. The ratio also does not test whether containers can be packed into particular bays.
- A stability change describes only the dimension-ratio indicator. It does not establish that a vessel is safe or stable at sea.
- A high modeled transformation cost follows from the numerical cost formula and scenario parameters; it is not an engineering energy estimate.
- The waste and recycling values are assumptions in the simulation. They do not measure actual waste prevention or recycling performance.
- The synthesis score is useful for tracing how the notebook combines its chosen metrics. Its weights and normalization constants are modeling choices, so the score should not be interpreted as a certified ranking or an optimality guarantee.

The main educational value is the connection between object state, rule application, recorded transformations, and aggregate reporting. A real stowage-planning model would require validated operational data and explicit assignment, capacity, compatibility, and safety constraints; those elements are outside this simulation's scope.

## Conclusion

This notebook demonstrates a procedural physical-cyber simulation for a vessel and a set of modeled containers. It represents material properties, geometry, compartments, environmental conditions, adaptation rules, and aggregate indicators as Python objects and methods.

The simulation's outputs are conditional on its stated assumptions and rules. In particular, programmable matter, energy quantities, waste avoidance, and recycling are illustrative concepts here, not verified capabilities or operational findings. The model does not perform container-to-slot assignment and does not establish a feasible or optimal vessel stowage plan.

The sequence of initialization, rule-based state changes, and metric calculation provides a foundation for understanding how a digital representation can track physical-system states. Any application to maritime operations would require empirically grounded data, validated physical models, and explicit operational and safety requirements.